# DSA210 Project - Football Market Value Analysis

In this project, I analyze how player performance and age relate to market value.

The dataset includes Premier League attacking players and variables such as age, minutes played, goals, assists, and market value.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

## 1. Loading the Dataset

First, I load the dataset and look at the first rows to understand its structure.

In [ ]:
possible_paths = [
    "yigitdsa210data.xlsx",
    "yigitdsa210data(1).xlsx",
    "/content/yigitdsa210data.xlsx",
    "/content/yigitdsa210data(1).xlsx",
    "/content/sample_data/yigitdsa210data.xlsx",
    "/content/sample_data/yigitdsa210data(1).xlsx",
    "/content/drive/MyDrive/yigitdsa210data.xlsx",
    "/content/drive/MyDrive/yigitdsa210data(1).xlsx"
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError("Dataset file was not found. Please upload yigitdsa210data.xlsx to Colab.")

df = pd.read_excel(data_path)
df.head()

In [ ]:
df.columns

## 2. Data Cleaning

The market value column is written as text, such as €18.00m. I convert it into a numeric value so that it can be used in plots, hypothesis testing, and machine learning.

In [ ]:
# Remove duplicate rate columns if they exist
# I keep the main total columns: Gls, Ast, and G+A.
df = df.drop(columns=["Gls.1", "Ast.1"], errors="ignore")

def convert_market_value(x):
    if pd.isna(x):
        return None

    x = str(x).strip().replace("€", "").lower()

    if x.endswith("m"):
        return float(x[:-1]) * 1_000_000
    elif x.endswith("k"):
        return float(x[:-1]) * 1_000
    else:
        return float(x)

# Convert market value into numeric form
df["Market Value Numeric"] = df["Market Value"].apply(convert_market_value)

# Convert numerical columns safely
numeric_cols = ["Age", "MP", "Starts", "Min", "Gls", "Ast", "G+A", "Market Value Numeric"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove rows with missing values in the columns used in this project
df = df.dropna(subset=["Age", "MP", "Min", "Gls", "Ast", "G+A", "Market Value Numeric"])

df[["Player", "Pos", "Age", "MP", "Min", "Gls", "Ast", "G+A", "Market Value", "Market Value Numeric"]].head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

The dataset is now cleaned. Market value is numeric, and the columns needed for the analysis do not have missing values.

## 3. Exploratory Data Analysis

I use multiple plots to see how age and performance variables relate to market value.

In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(df["Market Value Numeric"], bins=15)
plt.xlabel("Market Value")
plt.ylabel("Number of Players")
plt.title("Distribution of Market Value")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["Age"], df["Market Value Numeric"])
plt.xlabel("Age")
plt.ylabel("Market Value")
plt.title("Age vs Market Value")
plt.show()

From this graph, age alone does not seem to have a clear strong relationship with market value.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["Gls"], df["Market Value Numeric"])
plt.xlabel("Goals")
plt.ylabel("Market Value")
plt.title("Goals vs Market Value")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["Ast"], df["Market Value Numeric"])
plt.xlabel("Assists")
plt.ylabel("Market Value")
plt.title("Assists vs Market Value")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["G+A"], df["Market Value Numeric"])
plt.xlabel("Goals + Assists")
plt.ylabel("Market Value")
plt.title("Goals + Assists vs Market Value")
plt.show()

These plots give a better view of the dataset. I compare market value with age, goals, assists, and total goal contributions.

## 4. Hypothesis Testing

I test more than one hypothesis because market value may be related to different variables.

H1: Age is related to market value.  
H2: Goals are related to market value.  
H3: Goals + assists are related to market value.

In [ ]:
def correlation_test(x_col, y_col="Market Value Numeric"):
    temp = df[[x_col, y_col]].dropna()
    corr, p_value = pearsonr(temp[x_col], temp[y_col])
    print(f"{x_col} vs {y_col}")
    print("Correlation:", corr)
    print("P-value:", p_value)
    print()

correlation_test("Age")
correlation_test("Gls")
correlation_test("G+A")

The correlation results show which variables have a stronger relationship with market value.

If the p-value is lower than 0.05, the result is statistically significant. If it is higher than 0.05, the relationship is not statistically significant.

## 5. Machine Learning Model

I use a simple linear regression model to predict market value from age, minutes, goals, assists, and total goal contributions.

In [ ]:
features = ["Age", "Min", "Gls", "Ast", "G+A"]
X = df[features]
y = df["Market Value Numeric"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("R2 Score:", r2_score(y_test, predictions))
print("MAE:", mean_absolute_error(y_test, predictions))

In [ ]:
coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_
})

coefficients

The regression model gives a basic idea of how well age and performance variables can predict market value.

The model works, but market value is not only based on these variables. Other factors like team quality, league reputation, contract situation, and player popularity may also matter.

## Conclusion

Overall, this project shows that player market value is not explained by one factor only.

Age alone does not seem to strongly explain market value. Performance variables such as goals, assists, and total goal contributions are also important to check.

The machine learning model gives a simple prediction, but it is still limited. Market value can also depend on non-performance factors such as team, reputation, contract length, and transfer demand.